# Dask Delayed

In [ ]:
import random
import time

import dask
from bs4 import BeautifulSoup
import dask.bag as db

import json
import re

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/ФУППРФ/5 семестр/Технологии обработки больших данных/data

/content/drive/MyDrive/ФУППРФ/5 семестр/Технологии обработки больших данных/data


Материалы:
* Макрушин С.В. Лекция 13: Dask Delayed
* https://docs.dask.org/en/latest/delayed.html
* JESSE C. DANIEL. Data Science with Python and Dask.


## Задачи для совместного разбора

![](https://i.imgur.com/AwiN8y6.png)
![](https://i.imgur.com/ceY6guU.png)

1. Напишите 2 функции, имитирующие CPU-bound задачу и IO-bound задачу:

`cpu_task()`: генерирует 100 тыс. случайных чисел и возвращает их сумму (без использования `numpy`)

`io_task()`: "спит" 0.1 сек, затем генерирует случайное число и возвращает его

Замерьте время выполнения 100 последовательных вызовов каждой из этих функций. Распараллелив вычисления при помощи `dask.delayed`, сократите время выполнения. Исследуйте, как зависит время вычислений от выбранного планировщика `scheduler`.

In [ ]:
def cpu_task():
    return sum([random.randint(-1000, 1000) for _ in range(100_000)])

def io_task():
    time.sleep(0.1)
    return random.randint(-1000, 1000)

In [ ]:
%%time
for _ in range(100):
    cpu_task()

CPU times: user 10.5 s, sys: 35.9 ms, total: 10.6 s
Wall time: 10.8 s


In [ ]:
%%time
for _ in range(100):
    io_task()

CPU times: user 74 ms, sys: 12.4 ms, total: 86.4 ms
Wall time: 10 s


In [ ]:
@dask.delayed
def cpu_task():
    return sum([random.randint(-1000, 1000) for _ in range(100_000)])

@dask.delayed
def io_task():
    time.sleep(0.1)
    return random.randint(-1000, 1000)

In [ ]:
%%time
all_ops = []
for _ in range(100):
    all_ops.append(cpu_task())

_ = dask.compute(*all_ops)

CPU times: user 155 ms, sys: 17.2 ms, total: 173 ms
Wall time: 13 s


In [ ]:
%%time
all_ops = []
for _ in range(100):
    all_ops.append(io_task())

_ = dask.compute(*all_ops)

CPU times: user 95.6 ms, sys: 7.32 ms, total: 103 ms
Wall time: 5.87 s


In [ ]:
%%time
all_ops = []
for _ in range(100):
    all_ops.append(cpu_task())

_ = dask.compute(*all_ops, scheduler='synchronous')

CPU times: user 11.7 s, sys: 52.8 ms, total: 11.8 s
Wall time: 12 s


In [ ]:
%%time
all_ops = []
for _ in range(100):
    all_ops.append(io_task())

_ = dask.compute(*all_ops, scheduler='synchronous')

CPU times: user 116 ms, sys: 13.3 ms, total: 130 ms
Wall time: 10.1 s


In [ ]:
%%time
all_ops = []
for _ in range(100):
    all_ops.append(cpu_task())

_ = dask.compute(*all_ops, scheduler='threads')

CPU times: user 11.8 s, sys: 73 ms, total: 11.8 s
Wall time: 12 s


In [ ]:
%%time
all_ops = []
for _ in range(100):
    all_ops.append(io_task())

_ = dask.compute(*all_ops, scheduler='threads')

CPU times: user 88.6 ms, sys: 7.33 ms, total: 95.9 ms
Wall time: 5.06 s


In [ ]:
%%time
all_ops = []
for _ in range(100):
    all_ops.append(cpu_task())

_ = dask.compute(*all_ops, scheduler='processes')

CPU times: user 140 ms, sys: 15.3 ms, total: 155 ms
Wall time: 12.6 s


In [ ]:
%%time
all_ops = []
for _ in range(100):
    all_ops.append(io_task())

_ = dask.compute(*all_ops, scheduler='processes')

CPU times: user 87.1 ms, sys: 14.1 ms, total: 101 ms
Wall time: 5.83 s


In [ ]:
%%time
all_ops = []
for _ in range(100):
    all_ops.append(cpu_task())

_ = dask.compute(*all_ops, scheduler='multiprocessing')

CPU times: user 147 ms, sys: 13.8 ms, total: 160 ms
Wall time: 12.6 s


In [ ]:
%%time
all_ops = []
for _ in range(100):
    all_ops.append(io_task())

_ = dask.compute(*all_ops, scheduler='multiprocessing')

CPU times: user 441 ms, sys: 32 ms, total: 473 ms
Wall time: 5.87 s


## Лабораторная работа 14

1. Напишите функцию, которая считывает файл формата xml из архива `reviewers_full.zip` и по данным этого файла формирует список словарей, содержащих следующие ключи: `username`, `name`, `sex`, `country`, `mail`, `registered`, `birthdate`, `name_prefix`, `country_code`. Часть из этих значений в исходном файле хранится в виде тэгов, часть - в виде атрибутов тэгов. Для конкретного человека какие-то из этих ключей могут отсутствовать.



In [ ]:
def read_xml(file_name):
    with open(file_name) as f:
        xml_doc = BeautifulSoup(f, features="xml")

    users_list = []
    for user in xml_doc.find_all('user'):
        user_dict = {}
        if user.has_attr('prefix'):
            user_dict['user_prefix'] = user['prefix']

        for feature in user.findChildren():
            if feature.name == 'country':
                if feature.has_attr('code'):
                    user_dict['country_prefix'] = feature['code']
            user_dict[feature.name] = feature.text

        users_list.append(user_dict)

    return users_list

In [ ]:
read_xml('reviewers_full/reviewers_full_0.xml')[:3]

[{'user_prefix': 'Mrs.',
  'birthdate': '1988-01-25',
  'id': '556011',
  'sex': 'F',
  'username': 'gabrielacalhoun'},
 {'birthdate': '1985-01-19',
  'country_prefix': 'NO',
  'country': 'Norway',
  'id': '1251087',
  'mail': 'qware@gmail.com',
  'username': 'qbaxter'},
 {'birthdate': '1955-07-03',
  'id': '537188',
  'mail': 'stephaniestrong@yahoo.com',
  'name': 'Dana Moore',
  'registered': '2018-11-21',
  'username': 'crosschristopher'}]

2. Измерьте время выполнения функции из задания 1 на всех файлах из архива. Ускорьте время выполнения, используя `dask.delayed`.

In [ ]:
files = [f'reviewers_full/reviewers_full_{i}.xml' for i in range(5)]

In [ ]:
%%time
results = []
for file_ in files:
    results.append(read_xml(file_))


CPU times: user 1min 22s, sys: 896 ms, total: 1min 23s
Wall time: 1min 29s


In [ ]:
%%time

_ = [read_xml(file_) for file_ in files]

CPU times: user 1min 28s, sys: 983 ms, total: 1min 29s
Wall time: 1min 32s


In [ ]:
%%time
_ = dask.compute([dask.delayed(read_xml)(file_) for file_ in files], scheduler='processes')

CPU times: user 799 ms, sys: 217 ms, total: 1.02 s
Wall time: 1min 9s


In [ ]:
%%time
_ = dask.compute([dask.delayed(read_xml)(file_) for file_ in files], scheduler='multiprocessing')

CPU times: user 958 ms, sys: 148 ms, total: 1.11 s
Wall time: 1min 10s


3. Задекорируйте функцию из задания 1 при помощи `dask.delayed` и создайте список `reviewers`, состоящий из 5 объектов `delayed` (по одному объекту на файл). Из списка объектов `delayed`, создайте `dask.bag` при помощи метода `db.from_delayed`. Добавьте ключ `birth_year`, в котором хранится год рождения человека. Оставьте в выборке только тех людей, которые __наверняка__ моложе 1980 года. Преобразуйте поле `id` к целому типу.

In [ ]:
@dask.delayed
def read_xml(file_name):
    with open(file_name) as f:
        xml_doc = BeautifulSoup(f, features="xml")

    users_list = []
    for user in xml_doc.find_all('user'):
        user_dict = {}
        if user.has_attr('prefix'):
            user_dict['user_prefix'] = user['prefix']

        for feature in user.findChildren():
            if feature.name == 'country':
                if feature.has_attr('code'):
                    user_dict['country_prefix'] = feature['code']
            user_dict[feature.name] = feature.text

        users_list.append(user_dict)

    return users_list

In [ ]:
delayed_list = [read_xml(file_) for file_ in files]
users_bag = db.from_delayed(delayed_list)

In [ ]:
def add_birth_year(user):
    if 'birthdate' in user.keys():
        year_of_birth = int(user['birthdate'][:4])
        user['birth_year'] = year_of_birth
    return user

In [ ]:
users_bag = users_bag.map(add_birth_year)
users_bag.take(4)

({'user_prefix': 'Mrs.',
  'birthdate': '1988-01-25',
  'id': '556011',
  'sex': 'F',
  'username': 'gabrielacalhoun',
  'birth_year': 1988},
 {'birthdate': '1985-01-19',
  'country_prefix': 'NO',
  'country': 'Norway',
  'id': '1251087',
  'mail': 'qware@gmail.com',
  'username': 'qbaxter',
  'birth_year': 1985},
 {'birthdate': '1955-07-03',
  'id': '537188',
  'mail': 'stephaniestrong@yahoo.com',
  'name': 'Dana Moore',
  'registered': '2018-11-21',
  'username': 'crosschristopher',
  'birth_year': 1955},
 {'birthdate': '2007-04-30',
  'country_prefix': 'CU',
  'country': 'Cuba',
  'id': '250427',
  'mail': 'wjarvis@yahoo.com',
  'name': 'Jennifer Horne',
  'registered': '2013-11-20',
  'username': 'karen27',
  'birth_year': 2007})

In [ ]:
def is_younger_1980(user):
    if 'birth_year' in user.keys():
        if user['birth_year'] > 1980:
            return True
    return False

In [ ]:
users_bag = users_bag.filter(is_younger_1980)
users_bag.take(6)

({'user_prefix': 'Mrs.',
  'birthdate': '1988-01-25',
  'id': '556011',
  'sex': 'F',
  'username': 'gabrielacalhoun',
  'birth_year': 1988},
 {'birthdate': '1985-01-19',
  'country_prefix': 'NO',
  'country': 'Norway',
  'id': '1251087',
  'mail': 'qware@gmail.com',
  'username': 'qbaxter',
  'birth_year': 1985},
 {'birthdate': '2007-04-30',
  'country_prefix': 'CU',
  'country': 'Cuba',
  'id': '250427',
  'mail': 'wjarvis@yahoo.com',
  'name': 'Jennifer Horne',
  'registered': '2013-11-20',
  'username': 'karen27',
  'birth_year': 2007},
 {'user_prefix': 'Miss',
  'birthdate': '2005-03-29',
  'id': '452355',
  'name': 'Cynthia Johnson',
  'sex': 'F',
  'username': 'smullen',
  'birth_year': 2005},
 {'birthdate': '1983-02-20',
  'country_prefix': 'nan',
  'country': 'Namibia',
  'id': '2056668',
  'mail': 'claudiamccoy@gmail.com',
  'name': 'Michele Richards',
  'registered': '2009-08-12',
  'username': 'franklinhancock',
  'birth_year': 1983},
 {'birthdate': '2015-09-29',
  'country

In [ ]:
def id_to_int(user):
    user['id'] = int(user['id'])
    return user

In [ ]:
users_bag = users_bag.map(id_to_int)
users_bag.take(4)

({'user_prefix': 'Mrs.',
  'birthdate': '1988-01-25',
  'id': 556011,
  'sex': 'F',
  'username': 'gabrielacalhoun',
  'birth_year': 1988},
 {'birthdate': '1985-01-19',
  'country_prefix': 'NO',
  'country': 'Norway',
  'id': 1251087,
  'mail': 'qware@gmail.com',
  'username': 'qbaxter',
  'birth_year': 1985},
 {'birthdate': '2007-04-30',
  'country_prefix': 'CU',
  'country': 'Cuba',
  'id': 250427,
  'mail': 'wjarvis@yahoo.com',
  'name': 'Jennifer Horne',
  'registered': '2013-11-20',
  'username': 'karen27',
  'birth_year': 2007},
 {'user_prefix': 'Miss',
  'birthdate': '2005-03-29',
  'id': 452355,
  'name': 'Cynthia Johnson',
  'sex': 'F',
  'username': 'smullen',
  'birth_year': 2005})

4. Из `dask.bag`, полученного в задании 3, создайте `dask.dataframe` при помощи метода `bag.to_dataframe`. Укажите столбец `id` в качестве индекса.

In [ ]:
users_df = users_bag.to_dataframe()

In [ ]:
users_df.head()

,user_prefix,birthdate,id,sex,username,birth_year
0,Mrs.,1988-01-25,556011,F,gabrielacalhoun,1988
1,<NA>,1985-01-19,1251087,<NA>,qbaxter,1985
2,<NA>,2007-04-30,250427,<NA>,karen27,2007
3,Miss,2005-03-29,452355,F,smullen,2005
4,<NA>,1983-02-20,2056668,<NA>,franklinhancock,1983


In [ ]:
users_df = users_df.set_index('id')
users_df.head()

,user_prefix,birthdate,sex,username,birth_year
id,,,,,
1676,<NA>,1983-06-24,M,lgeorge,1983
1792,<NA>,1986-03-12,F,qbeard,1986
1938,<NA>,1991-11-11,<NA>,adambrown,1991
2046,<NA>,1981-11-27,F,vthompson,1981
2095,Mrs.,1984-09-23,F,djohnson,1984


5. Назовем отзыв негативным, если оценка равна 0, 1 или 2. Загрузите данные о негативных отзывах из файлов архива `reviews_full` (__ЛР12__) в виде `dask.DataFrame`. Посчитайте количество отзывов с группировкой по пользователю, оставившему отзыв. Объедините результат с таблицей, полученной в задаче 4.

In [ ]:
def parse_json(tup):
    json_, file_name = tup
    d = json.loads(json_)
    d['rating'] = int(re.findall(r'reviews_(\d)+', file_name)[0])
    return d

records = db.read_text([f'12/reviews_{i}.json' for i in range(0, 3)], include_path=True)
records

dask.bag<bag-from-delayed, npartitions=3>

In [ ]:
records_parsed = records.map(parse_json)

In [ ]:
reviews_df = records_parsed.to_dataframe()
reviews_df

,user_id,recipe_id,date,review,rating
npartitions=3,,,,,
,int64,int64,string,string,int64
,...,...,...,...,...
,...,...,...,...,...
,...,...,...,...,...


In [ ]:
user_count = reviews_df[['user_id', 'recipe_id']].groupby('user_id').count().rename(columns={'recipe_id':'n_reviews'})
user_count

,n_reviews
npartitions=1,
,int64
,...


In [ ]:
combined = users_df.join(user_count)

In [ ]:
combined.head()

,user_prefix,birthdate,sex,username,birth_year,n_reviews
id,,,,,,
1676,<NA>,1983-06-24,M,lgeorge,1983,29.0
1792,<NA>,1986-03-12,F,qbeard,1986,14.0
1938,<NA>,1991-11-11,<NA>,adambrown,1991,3.0
2046,<NA>,1981-11-27,F,vthompson,1981,3.0
2095,Mrs.,1984-09-23,F,djohnson,1984,NaN
